In [ ]:
import torch

from test_attn_order_eval import AttnOrderEval

In [ ]:
def generate_masked_attn(a, b, value_mask = -1):
    a = a.detach().clone()

    for i in range(b.size()[0]):
        a[i, b[:i]] = -1
    # end
    return a
# end

if __name__ == "__main__":
    torch.manual_seed(0)
    folder_base = 'stats/0'
    T = L = 64

    pos_base = 32 + 64 + 64 + 64

    unmask = torch.load(f'{folder_base}/unmask_{pos_base}_{pos_base + L}.pt')
    unmask = unmask.squeeze(-1) - pos_base
    order = unmask

    step_of = torch.full((L,), -1, dtype=torch.long)
    step_of[order] = torch.arange(T)
    gap = step_of.view(1, L) - torch.arange(T).view(T, 1)          # [T, L]

    attn_perfect = torch.where(gap > 0, 1.0 / gap.clamp_min(1).double(), torch.zeros(1, dtype=torch.float64))
    attn_random = torch.rand(T, L, dtype=torch.float64)

    attn = torch.load(f'{folder_base}/attn_32_96.pt')
    attn_last = attn[:,-1,:,:].squeeze(1) # -> [step, attn]
    attn_last_switched = attn_last[unmask, :]

    for name, attn in [("perfect", attn_perfect), ("random ", attn_random), ("attn", attn_last_switched)]:
        ev = AttnOrderEval(attn, order)
        r = ev.recall_at_h(5).nanmean().item()
        p = ev.pr_auc(5).nanmean().item()
        nd = ev.ndcg_at_h(10).nanmean().item()
        print("{}  recall@5={:.3f}  pr_auc@5={:.3f}  ndcg@10={:.3f}".format(name, r, p, nd))
    # end

perfect  recall@5=1.000  pr_auc@5=1.000  ndcg@10=1.000
random   recall@5=0.251  pr_auc@5=0.322  ndcg@10=0.402
attn  recall@5=0.668  pr_auc@5=0.769  ndcg@10=0.809
